# 04 - Linear lap-time model

Notebook 03 showed that baselines which ignore the lap number barely beat predicting
"no change". Hypothesis: fuel burn is the missing piece. This notebook adds a fuel
term and gives each compound its own wear slope, then checks whether that hypothesis
holds. Same setup as before: train 2022-2023, score on 2024, target `delta_s`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from f1lab.features import prepare, split, mae, rmse
from f1lab.models import linear_features, make_linear, raw_coefficients

raw = pd.read_parquet("../data/laps.parquet")
clean = pd.read_parquet("../data/laps_clean.parquet")
train, val = split(prepare(raw, clean), "dev")
tracks = sorted(train["track"].unique())
FUEL = ["fuel_laps", "fuel_frac"]

def evaluate(name, robust, drop=(), with_tracks=False):
    A = linear_features(train, tracks if with_tracks else None).drop(columns=list(drop))
    B = linear_features(val, tracks if with_tracks else None).drop(columns=list(drop))
    model = make_linear(robust).fit(A, train["delta_s"])
    pred = pd.Series(model.predict(B), index=val.index)
    return {"model": name, "MAE (s)": mae(val["delta_s"], pred), "RMSE (s)": rmse(val["delta_s"], pred)}, model, A, pred

rows = [
    evaluate("OLS, no fuel term", False, drop=FUEL)[0],
    evaluate("OLS, fuel = laps remaining", False, drop=["fuel_frac"])[0],
    evaluate("OLS, fuel = fraction of race left", False, drop=["fuel_laps"])[0],
    evaluate("OLS, both fuel terms", False)[0],
    evaluate("Huber, both fuel terms", True)[0],
    evaluate("Huber, both fuel terms + track effects", True, with_tracks=True)[0],
]
pd.DataFrame(rows).set_index("model").round(3)

,MAE (s),RMSE (s)
model,,
"OLS, no fuel term",0.848,1.336
"OLS, fuel = laps remaining",0.686,1.193
"OLS, fuel = fraction of race left",0.643,1.163
"OLS, both fuel terms",0.638,1.161
"Huber, both fuel terms",0.629,1.161
"Huber, both fuel terms + track effects",0.631,1.166


**Hypothesis confirmed.** Adding any fuel term takes MAE from about 0.85 to between
0.69 and 0.64, far better than any baseline (best was 0.855). The fraction of the race
remaining beats raw laps remaining, which makes sense: fuel burned per lap depends on
how long the race is. A robust (Huber) fit shaves a little more off (0.629), since
incident laps no longer drag it around. Adding a track effect does not help (0.631), in
line with the earlier finding that per-track effects are noisy from two seasons of data.

## What the model learned

In [2]:
metrics, model, A, pred = evaluate("Huber, both fuel terms", True)
coef = raw_coefficients(model, list(A.columns)).round(4)
coef

fuel_laps       -0.0229
fuel_frac        4.3209
offset_SOFT     -0.1851
wear_SOFT        0.0473
offset_MEDIUM   -0.0512
wear_MEDIUM      0.0389
offset_HARD      0.1425
wear_HARD        0.0245
track_temp       0.0011
dtype: float64

In [3]:
# net fuel effect per lap of race remaining, for a short and a long race
for laps in (50, 57, 78):
    print(f"{laps}-lap race: {coef['fuel_frac'] / laps + coef['fuel_laps']:+.3f} s per lap remaining")

50-lap race: +0.064 s per lap remaining
57-lap race: +0.053 s per lap remaining
78-lap race: +0.032 s per lap remaining


Reading the coefficients (seconds):
- **Fuel.** The two fuel terms work together: for a 57-lap race each extra lap of fuel
  costs about 0.05 s, for the 78-lap Monaco about 0.03 s. A heavier car is slower, as
  expected, and the size is plausible.
- **Tyre wear per lap:** SOFT 0.047, MEDIUM 0.039, HARD 0.025. Softer compounds wear
  faster, the order physics predicts. Notebook 02's pooled estimate had HARD above
  MEDIUM; the fuel fraction and robust loss appear to fix that, though that was a
  different setup so I would not read too much into the comparison.
- **Compound offset at zero age:** SOFT -0.19 s, MEDIUM -0.05, HARD +0.14. About 0.33 s
  between soft and hard on fresh tyres.
- **Track temperature:** about +0.001 s per degree, negligible, consistent with
  notebook 02.

## Where it is still wrong

In [4]:
err = (val["delta_s"] - pred).abs()
stage = pd.cut(1 - val["laps_remaining"] / val["race_laps"], [0, .25, .5, .75, 1])
print("MAE by race stage:"); print(err.groupby(stage, observed=True).mean().round(3).to_string())
print("\nMAE by compound:"); print(err.groupby(val["compound"]).mean().round(3).to_string())
unseen = val["track"] == "Shanghai"
print(f"\nMAE at Shanghai (no training data): {err[unseen].mean():.3f} | everywhere else: {err[~unseen].mean():.3f}")

MAE by race stage:
(0.0, 0.25]    0.651
(0.25, 0.5]    0.529
(0.5, 0.75]    0.528
(0.75, 1.0]    0.798

MAE by compound:
compound
HARD      0.620
MEDIUM    0.640
SOFT      0.655

MAE at Shanghai (no training data): 0.904 | everywhere else: 0.618


Error is highest in the last quarter of the race (0.80 s against 0.53 to 0.65 earlier)
and at Shanghai, a track the model never saw (0.90 against 0.62). I have not tested
why. Candidate explanations: late-race traffic and lapped cars, engine and tyre
management at the end, tyre wear that is not linear (a cliff), and wear differing
by track. 0.63 s is still large next to the effects we are trying to model. The next
step is a non-linear model (gradient boosting) that can learn interactions, compared
against this one on the same data, and then looking at which of these explanations
the errors support.